# Concrete Compressive Strength Prediction

Predicts 28-day concrete compressive strength (MPa) from mix components, to support a smart concrete quote / mix-design assistant (e.g. for a ready-mix concrete company).

**Pipeline:** load & clean data → EDA → log-transform skewed features → train/test split → scale (fit on train only) → train Gradient Boosting model → evaluate (R², RMSE, MAE, cross-validation) → save model → predict on a new mix.

## 1. Load data

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("niteshyadav3103/concrete-compressive-strength")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv("/content/Concrete Compressive Strength.csv")
data.columns = data.columns.str.strip()
data.head()

In [ ]:
print("Shape of the data:", data.shape)

In [ ]:
data.info()

## 2. Clean data

In [ ]:
print("Number of duplicate rows:", data.duplicated().sum())
data.drop_duplicates(inplace=True)
print("Number of duplicate rows after cleaning:", data.duplicated().sum())

In [ ]:
data.describe()

## 3. Explore the data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle('Distributions of Concrete Dataset Variables (Raw)', fontsize=16)
axes = axes.flatten()
for i, col in enumerate(data.columns):
    sns.histplot(data[col], kde=True, ax=axes[i], color='skyblue')
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('distributions_raw.png')
print("Image saved as distributions_raw.png")

Several columns (Cement, Blast Furnace Slag, Fly Ash, Superplasticizer, Age) are right-skewed, with many zeros and a long tail. A `log1p` transform makes their distributions more symmetric, which helps tree-based models split more evenly. This transform is deterministic (it only looks at each row's own value), so applying it before the train/test split does **not** leak information between the two sets — unlike scaling, which we fit after the split below.

In [ ]:
cols_to_transform = [
    'Cement',
    'Blast Furnace Slag',
    'Fly Ash',
    'Superplasticizer',
    'Age (day)'
]

data[cols_to_transform] = np.log1p(data[cols_to_transform])

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle('Distributions of Concrete Dataset Variables (After log1p)', fontsize=16)
axes = axes.flatten()
for i, col in enumerate(data.columns):
    sns.histplot(data[col], kde=True, ax=axes[i], color='skyblue')
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('distributions_log.png')
print("Image saved as distributions_log.png")

In [ ]:
corr_matrix = data.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Matrix (After log1p Transform)', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_matrix.png')
print("Image saved as correlation_matrix.png")

## 4. Split, then scale (fit only on train)

Scaling learns a mean and standard deviation from the data, so if we fit it before splitting, information from the test set leaks into training and the reported score becomes optimistic. We split **first**, then fit `StandardScaler` on the training set only, and use that same fitted scaler to transform the test set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = data.drop('Concrete compressive strength', axis=1)
y = data['Concrete compressive strength']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

## 5. Train the model

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr_model = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    min_samples_split=4,
    random_state=42
)

gbr_model.fit(X_train_scaled, y_train)
y_pred_gbr = gbr_model.predict(X_test_scaled)

## 6. Evaluate

R² alone doesn't say how far off the predictions are in real units. We add RMSE and MAE (in MPa, the unit engineers actually care about), plus 5-fold cross-validation so the R² score isn't just a product of one lucky train/test split.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline

r2_gbr = r2_score(y_test, y_pred_gbr)
rmse_gbr = np.sqrt(mean_squared_error(y_test, y_pred_gbr))
mae_gbr = mean_absolute_error(y_test, y_pred_gbr)

print(f"R-squared: {r2_gbr:.4f}")
print(f"RMSE: {rmse_gbr:.2f} MPa")
print(f"MAE:  {mae_gbr:.2f} MPa")

# Cross-validation with scaling + model in one pipeline, so each fold scales using only its own
# training portion — the correct way to cross-validate when a preprocessing step is fit on data.
#
# shuffle=True matters here: cross_val_score's default KFold (when you just pass an integer to
# `cv`) does NOT shuffle, so folds are cut from the data in its original row order. On this
# dataset that produced wildly unstable folds (R² ranging from -0.80 to 0.85, mean 0.45) —
# not because the model was bad, but because rows sharing similar properties were grouped
# together in the raw file. Building an explicit KFold(shuffle=True) fixes this.
cv_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=5,
        min_samples_split=4, random_state=42
    ))
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(cv_pipeline, X, y, cv=cv, scoring='r2')

print(f"\n5-fold CV R-squared: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"Fold scores: {np.round(cv_scores, 4)}")

R-squared: 0.9434
RMSE: 4.11 MPa
MAE:  2.60 MPa

5-fold CV R-squared: 0.9372 (+/- 0.0081)
Fold scores: [0.9434 0.9428 0.9451 0.9279 0.9269]


**Why this took two tries.** The first version of this cell passed `cv=5` directly to `cross_val_score`. For regression, that builds a plain `KFold` with `shuffle=False`, so each fold is a contiguous slice of the file in its original row order. On this dataset that produced unstable, misleading scores: a mean R² of only 0.45 with one fold scoring **-0.80** — which looked like severe overfitting or a broken model.

Rather than trust that number, the fix was to build an explicit `KFold(n_splits=5, shuffle=True, random_state=42)` and pass that to `cross_val_score` instead of a bare integer. That randomizes which rows land in which fold before splitting. The result: **CV R² = 0.9372 (± 0.0081)**, with every fold landing between 0.927 and 0.945 — close to, and consistent with, the 0.9434 R² from the single train/test split.

**Takeaway:** the model was fine all along; the first cross-validation setup wasn't. A large, unstable CV spread (especially a negative fold) is a signal to check *how* the folds were built — row order, grouping, or a missing shuffle — before concluding the model itself is unreliable.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Actual vs Predicted
axes[0].scatter(y_test, y_pred_gbr, alpha=0.6, color='#1f77b4')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_title('Actual vs. Predicted Strength', fontsize=12)
axes[0].set_xlabel('Actual Strength (MPa)')
axes[0].set_ylabel('Predicted Strength (MPa)')

# 2. Residual Plot
residuals = y_test - y_pred_gbr
sns.histplot(residuals, kde=True, ax=axes[1], color='#9467bd')
axes[1].set_title('Residuals Distribution (Errors)', fontsize=12)
axes[1].set_xlabel('Error (Actual - Predicted)')
axes[1].set_ylabel('Frequency')

# 3. Feature Importance
importance = gbr_model.feature_importances_
sorted_idx = np.argsort(importance)
axes[2].barh(X.columns[sorted_idx], importance[sorted_idx], color='#2ca02c')
axes[2].set_title('Feature Importance', fontsize=12)
axes[2].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('regression_evaluation.png')
print("Image saved as regression_evaluation.png")

## 7. Save the model

In [ ]:
import joblib

joblib.dump(gbr_model, 'concrete_model.pkl')
joblib.dump(scaler, 'concrete_scaler.pkl')

print("Model and scaler saved successfully.")

## 8. Predict on a new mix

In [ ]:
loaded_model = joblib.load('concrete_model.pkl')
loaded_scaler = joblib.load('concrete_scaler.pkl')

real_mix = pd.DataFrame({
    'Cement': [300.0],
    'Blast Furnace Slag': [0.0],
    'Fly Ash': [0.0],
    'Water': [190.0],
    'Superplasticizer': [0.0],
    'Coarse Aggregate': [1000.0],
    'Fine Aggregate': [800.0],
    'Age (day)': [28.0]
})

cols_to_transform = ['Cement', 'Blast Furnace Slag', 'Fly Ash', 'Superplasticizer', 'Age (day)']
real_mix[cols_to_transform] = np.log1p(real_mix[cols_to_transform])
real_mix_scaled = loaded_scaler.transform(real_mix)
real_mix_scaled_df = pd.DataFrame(real_mix_scaled, columns=real_mix.columns)

prediction = loaded_model.predict(real_mix_scaled_df)
print(f"Predicted concrete compressive strength: {prediction[0]:.2f} MPa")